# Feasibility metric comparison — 2-tri vs forward-diff (shoelace) vs central-diff Jdet

Three feasibility metrics for 2D deformation, ordered from **strictest** to **most lenient**:

| metric | per-cell formula | semantic |
|---|---|---|
| **2-triangle** (manuscript) | `T1, T2` each ≥ thr | both halves of the TR-BL split are non-folded |
| **shoelace** (forward-diff) | `T1 + T2 = signed quad area` ≥ thr | net warped-quad area non-negative |
| **central-diff Jdet** | 4-point determinant of central differences ≥ thr | local derivative determinant positive |

All three reduce to the *same* mathematical idea (no folding) but they measure it differently — and on the manuscript's dense slices they give **different feasibility verdicts**.

This notebook runs the two existing full-grid barriers on a worst-case slice (z=12) and measures *all three* metrics on each output, so we can see exactly where they agree and where they disagree.

In [ ]:
import os, sys, time
sys.path.insert(0, os.path.abspath('../..'))

import numpy as np
import matplotlib.pyplot as plt

from dvfopt.core.iterative2d_barrier import iterative_2d_barrier
from dvfopt.core.iterative2d_tri_barrier import iterative_2d_tri_barrier
from dvfopt.jacobian.numpy_jdet import jacobian_det2D
from dvfopt.jacobian.triangle_sign import _triangle_areas_2d
from dvfopt.jacobian.shoelace import _shoelace_areas_2d

THRESHOLD = 0.01
LAM = (1.0, 10.0, 100.0, 1e3, 1e4, 1e5, 1e6)
MU = (1e-1, 1e-2, 1e-3)
MAX_ITER = 200

phi_full = np.load(os.path.abspath(os.path.join(
    '..', '..', 'data', 'corrected_correspondences_count_touching',
    'registered_output', 'deformation3d.npy')))
Z = 12
phi0 = np.stack([phi_full[1, Z].copy(), phi_full[2, Z].copy()])
H, W = phi0.shape[1], phi0.shape[2]
print(f'z={Z}: phi.shape = {phi0.shape}')

In [ ]:
def compute_metrics(phi2):
    """Return dict with 2D maps and (n_neg, min) for each metric."""
    T1, T2 = _triangle_areas_2d(phi2[0], phi2[1])
    tri = np.minimum(T1, T2)                 # 2-tri: min of the two
    sho = _shoelace_areas_2d(phi2[0], phi2[1])   # = T1 + T2
    j = np.squeeze(jacobian_det2D(phi2))
    return dict(
        tri_map=tri,
        sho_map=sho,
        jdet_map=j,
        tri_neg=int((tri <= 0).sum()),
        tri_min=float(tri.min()),
        sho_neg=int((sho <= 0).sum()),
        sho_min=float(sho.min()),
        jdet_neg=int((j <= 0).sum()),
        jdet_min=float(j.min()),
    )

def fmt(d, key):
    return f'n_neg={d[key + "_neg"]:5d}  min={d[key + "_min"]:+.5f}'

init = compute_metrics(phi0)
print(f'  init:   2-tri  {fmt(init,"tri")}')
print(f'          shoel  {fmt(init,"sho")}')
print(f'          jdet   {fmt(init,"jdet")}')

In [ ]:
print('--- Jdet (central-diff) barrier ---')
t0 = time.time()
out_j = iterative_2d_barrier(
    phi_full[:, Z:Z+1].copy(), threshold=THRESHOLD, margin=1e-3,
    lam_schedule=LAM, mu_schedule=MU,
    max_minimize_iter=MAX_ITER, windowed=False, verbose=0)
out_j = np.asarray(out_j)
phi_j = (np.stack([out_j[0], out_j[1]]) if out_j.ndim == 3 and out_j.shape[0] == 2
         else np.stack([out_j[1, 0], out_j[2, 0]]))
t_j = time.time() - t0
after_j = compute_metrics(phi_j)
print(f'  ({t_j:.0f}s)')
print(f'  2-tri  {fmt(after_j,"tri")}')
print(f'  shoel  {fmt(after_j,"sho")}')
print(f'  jdet   {fmt(after_j,"jdet")}')

print('\n--- 2-tri barrier ---')
t0 = time.time()
phi_t = iterative_2d_tri_barrier(
    phi0.copy(), threshold=THRESHOLD, margin=1e-3,
    lam_schedule=LAM, mu_schedule=MU,
    max_minimize_iter=MAX_ITER, anchor='l2', verbose=0)
t_t = time.time() - t0
after_t = compute_metrics(phi_t)
print(f'  ({t_t:.0f}s)')
print(f'  2-tri  {fmt(after_t,"tri")}')
print(f'  shoel  {fmt(after_t,"sho")}')
print(f'  jdet   {fmt(after_t,"jdet")}')

## Summary table — feasibility under each metric

In [ ]:
import pandas as pd
rows = []
for label, d in [('init', init), ('after Jdet barrier', after_j),
                  ('after 2-tri barrier', after_t)]:
    rows.append({
        'state': label,
        '2tri n_neg': d['tri_neg'],   '2tri min': d['tri_min'],
        '2tri feas': d['tri_neg'] == 0 and d['tri_min'] >= THRESHOLD - 1e-4,
        'shoe n_neg': d['sho_neg'],   'shoe min': d['sho_min'],
        'shoe feas': d['sho_neg'] == 0 and d['sho_min'] >= THRESHOLD - 1e-4,
        'jdet n_neg': d['jdet_neg'], 'jdet min': d['jdet_min'],
        'jdet feas': d['jdet_neg'] == 0 and d['jdet_min'] >= THRESHOLD - 1e-4,
    })
pd.DataFrame(rows).set_index('state')

## Visualization — the 3 metrics × 3 states

**Rows:** init / after Jdet barrier / after 2-tri barrier.
**Columns:** the metric value at each cell (2-tri, shoelace, central-diff Jdet).

Red = below 0 (folded by that metric). Color scale is symmetric and **shared per column** so the 3 stages of each metric are directly comparable. Folded cells (under each metric) are outlined in cyan.

In [ ]:
states = [('init', init), ('after Jdet barrier', after_j),
          ('after 2-tri barrier', after_t)]
metrics = [('2-triangle min(T1,T2)', 'tri_map', 'tri_neg', 'tri_min'),
           ('shoelace (forward-diff)', 'sho_map', 'sho_neg', 'sho_min'),
           ('central-diff Jdet', 'jdet_map', 'jdet_neg', 'jdet_min')]

# Per-column shared vmax (so rows are directly comparable per metric).
vmax_per_metric = []
for _, key, _, _ in metrics:
    vals = np.concatenate([d[key].ravel() for _, d in states])
    vmax_per_metric.append(max(abs(vals.min()), abs(vals.max()), 0.05))

fig, axes = plt.subplots(3, 3, figsize=(16, 13.5), constrained_layout=True)
for i, (state_label, d) in enumerate(states):
    for j, (metric_label, key, neg_key, min_key) in enumerate(metrics):
        ax = axes[i, j]
        m = d[key]
        vmax = vmax_per_metric[j]
        im = ax.imshow(m, cmap='RdBu_r', vmin=-vmax, vmax=vmax)
        # Fold contour
        ax.contour((m <= 0).astype(float), levels=[0.5],
                   colors='cyan', linewidths=0.4)
        ax.set_title(
            f'{state_label}\n{metric_label}: '
            f'n_neg={d[neg_key]}  min={d[min_key]:+.3f}',
            fontsize=9)
        ax.set_xticks([]); ax.set_yticks([])
        fig.colorbar(im, ax=ax, shrink=0.85)
fig.suptitle(f'z={Z}: three feasibility metrics × three corrected states  '
             '(cyan contour = folded under each metric)', fontsize=11)
plt.show()

In [ ]:
# Companion: per-metric histogram, all 3 states overlaid.
fig, axes = plt.subplots(1, 3, figsize=(16, 4.3), constrained_layout=True)
for j, (metric_label, key, _, _) in enumerate(metrics):
    ax = axes[j]
    bins = np.linspace(-0.05, 0.05, 70)
    for color, (state_label, d) in zip(['#888', '#c62828', '#1b8a3a'], states):
        ax.hist(d[key].ravel(), bins=bins, histtype='step',
                color=color, label=state_label, linewidth=1.5)
    ax.axvline(0, color='k', lw=0.7, label='T=0' if j == 0 else None)
    ax.axvline(THRESHOLD, color='#1b8a3a', ls='--', lw=1.0,
               label=f'thr {THRESHOLD}' if j == 0 else None)
    ax.set_yscale('log')
    ax.set_xlabel('metric value (zoom near 0)')
    ax.set_title(metric_label, fontsize=10)
    ax.grid(alpha=0.3)
    if j == 0:
        ax.legend(fontsize=8, loc='upper left')
fig.suptitle('Per-metric distribution at each stage (zoom near 0)', fontsize=11)
plt.show()

## Reading the figures

What the rows and columns reveal on z=12:

- **Init row** — three faces of the same severe folding. The 2-tri map has the largest red region (most strict), the shoelace map is intermediate (cells where the *net* quad is folded), the central-diff Jdet has the smallest red region (least strict).
- **Jdet-barrier row** — the rightmost column is *uniformly blue with no red* (`n_neg = 0`, `min Jdet = +0.011`): Jdet is feasible. But look left: the 2-tri map is *worse than the input* — central-diff Jdet's minimum is the average of the four corner determinants, so it can be positive while one of the two triangle halves of the same cell is severely negative. The barrier exploits this loophole.
- **2-tri barrier row** — leftmost column shows the 2-tri map sitting just below 0 in a thin band (the wall). The shoelace map mostly recovers (close to feasible since T1+T2 is small at the wall but the *sign* depends on which triangle is more negative). The Jdet map still has residual reds: pushing both halves of a cell to ≥0 simultaneously *constrains* the field more, and that extra constraint can pull central-diff Jdet negative elsewhere.

**Implication for the manuscript.** The 2-tri criterion is strictly stronger than central-diff Jdet — fields can be Jdet-feasible but 2-tri-pathological. On the densest slices the 2-tri-feasible set with bounded L2 correction appears to be (near-)empty. The manuscript's choice of 2-tri is therefore both more conservative and (for these dense fields) more demanding than the data admits with the L2-minimum principle.